# Data Preprocessing — Video Game Sales

Notebook ini berisi tahapan preprocessing data berdasarkan temuan dari EDA.

| Tahap | Keterangan |
|-------|------------|
| 1 | Import Library & Load Data |
| 2 | Kondisi Data Awal |
| 3 | Menghapus Missing Value |
| 4 | Menghapus Data Duplikat |
| 5 | Mengatasi Data Tidak Konsisten |
| 6 | Mengubah Format Data |
| 7 | Feature Engineering |
| 8 | Filtering Data Tidak Relevan |
| 9 | Hasil Akhir & Export |

## 1. Import Library & Load Data

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset mentah
df = pd.read_csv("../data/raw/video_game_sales.csv")
print("Dataset berhasil dimuat.")
print(f"Jumlah baris : {df.shape[0]:,}")
print(f"Jumlah kolom : {df.shape[1]}")

Dataset berhasil dimuat.
Jumlah baris : 16,598
Jumlah kolom : 11


## 2. Kondisi Data Awal

Menampilkan kondisi dataset sebelum preprocessing sebagai **baseline** perbandingan.

In [6]:
print("=" * 50)
print("KONDISI DATA AWAL")
print("=" * 50)
print(f"Jumlah baris    : {df.shape[0]:,}")
print(f"Jumlah kolom    : {df.shape[1]}")
print(f"Jumlah duplikat : {df.duplicated().sum()}")
print()
print("Missing Value per Kolom:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Jumlah Missing': missing, 'Persentase (%)': missing_pct})
print(missing_df[missing_df['Jumlah Missing'] > 0])
print()
print("Tipe Data:")
print(df.dtypes)

KONDISI DATA AWAL
Jumlah baris    : 16,598
Jumlah kolom    : 11
Jumlah duplikat : 0

Missing Value per Kolom:
           Jumlah Missing  Persentase (%)
Year                  271            1.63
Publisher              58            0.35

Tipe Data:
Rank              int64
Name                str
Platform            str
Year            float64
Genre               str
Publisher           str
NA_Sales        float64
EU_Sales        float64
JP_Sales        float64
Other_Sales     float64
Global_Sales    float64
dtype: object


**Temuan dari EDA:**
- Kolom `Year` memiliki **271 missing value** (1.63%) — bertipe float, seharusnya integer
- Kolom `Publisher` memiliki **58 missing value** (0.35%)
- Perlu dicek adanya duplikat
- Perlu dicek konsistensi nilai `Year` (ada tahun yang tidak wajar)

## 3. Menghapus Missing Value

Berdasarkan EDA, kolom yang memiliki missing value adalah `Year` (271 baris) dan `Publisher` (58 baris).
Karena jumlahnya kecil dibanding total data (16.598 baris), kita hapus baris tersebut.

In [5]:
df_clean = df.copy()

sebelum = len(df_clean)

# Hapus baris yang Year atau Publisher kosong
df_clean = df_clean.dropna(subset=['Year', 'Publisher'])

sesudah = len(df_clean)
terhapus = sebelum - sesudah

print("Missing value berhasil dihapus.")
print(f"Baris sebelum : {sebelum:,}")
print(f"Baris terhapus: {terhapus:,}")
print(f"Baris sesudah : {sesudah:,}")
print()
print("Sisa missing value:")
print(df_clean.isnull().sum())

Missing value berhasil dihapus.
Baris sebelum : 16,598
Baris terhapus: 307
Baris sesudah : 16,291

Sisa missing value:
Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64


## 4. Menghapus Data Duplikat

Mengecek dan menghapus baris yang benar-benar identik di semua kolom.

In [7]:
sebelum = len(df_clean)
jumlah_duplikat = df_clean.duplicated().sum()

print(f"Jumlah duplikat ditemukan: {jumlah_duplikat}")

if jumlah_duplikat > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"{jumlah_duplikat} duplikat berhasil dihapus.")
else:
    print("Tidak ada duplikat. Data tetap utuh.")

print(f"Baris sebelum : {sebelum:,}")
print(f"Baris sesudah : {len(df_clean):,}")

Jumlah duplikat ditemukan: 0
Tidak ada duplikat. Data tetap utuh.
Baris sebelum : 16,291
Baris sesudah : 16,291


## 5. Mengatasi Data Tidak Konsisten

Mengecek nilai-nilai yang tidak wajar atau tidak konsisten di tiap kolom.

In [ ]:
# 5a. Cek nilai Year yang tidak wajar 
print("Nilai unik Year (luar rentang 1980–2016):")
tahun_aneh = df_clean[~df_clean['Year'].between(1980, 2016)]['Year'].unique()
print(tahun_aneh if len(tahun_aneh) > 0 else "Tidak ada.")

# Hapus baris dengan tahun di luar rentang wajar
sebelum = len(df_clean)
df_clean = df_clean[df_clean['Year'].between(1980, 2016)]
print(f"\nBaris terhapus karena Year tidak wajar: {sebelum - len(df_clean)}")

Nilai unik Year (luar rentang 1980–2016):
[2020. 2017.]

Baris terhapus karena Year tidak wajar: 4


In [ ]:
# 5b. Cek konsistensi kolom teks (strip whitespace & title case)
kolom_teks = ['Name', 'Platform', 'Genre', 'Publisher']

for col in kolom_teks:
    df_clean[col] = df_clean[col].str.strip()          # hapus spasi di awal/akhir
    df_clean[col] = df_clean[col].str.replace(r'\s+', ' ', regex=True)  # hapus spasi ganda

print("Whitespace dan spasi ganda pada kolom teks berhasil dibersihkan.")
print(f"Kolom yang dibersihkan: {kolom_teks}")

Whitespace dan spasi ganda pada kolom teks berhasil dibersihkan.
Kolom yang dibersihkan: ['Name', 'Platform', 'Genre', 'Publisher']


In [ ]:
# 5c. Cek nilai Global_Sales vs penjumlahan regional
# Global_Sales seharusnya mendekati NA + EU + JP + Other
df_clean['_sales_check'] = (df_clean['NA_Sales'] + df_clean['EU_Sales'] +
                             df_clean['JP_Sales'] + df_clean['Other_Sales']).round(2)
df_clean['_selisih'] = (df_clean['Global_Sales'] - df_clean['_sales_check']).abs()

# Toleransi selisih pembulatan: <= 0.05
tidak_konsisten = df_clean[df_clean['_selisih'] > 0.05]
print(f"Baris dengan Global_Sales tidak konsisten (selisih > 0.05): {len(tidak_konsisten)}")

if len(tidak_konsisten) > 0:
    print("Contoh data tidak konsisten:")
    print(tidak_konsisten[['Name', 'NA_Sales', 'EU_Sales', 'JP_Sales',
                             'Other_Sales', 'Global_Sales', '_selisih']].head())

# Hapus kolom bantu
df_clean = df_clean.drop(columns=['_sales_check', '_selisih'])
print("\nPengecekan konsistensi Global_Sales selesai.")

Baris dengan Global_Sales tidak konsisten (selisih > 0.05): 0

Pengecekan konsistensi Global_Sales selesai.


## 6. Mengubah Format Data

Mengubah tipe data kolom agar sesuai dengan nilainya.

In [ ]:
# 6a. Ubah Year dari float ke integer
print("Tipe data sebelum:")
print(df_clean[['Year']].dtypes)

df_clean['Year'] = df_clean['Year'].astype(int)

print("\nTipe data sesudah:")
print(df_clean[['Year']].dtypes)
print("\nKolom Year berhasil diubah dari float64 → int64.")

Tipe data sebelum:
Year    int64
dtype: object

Tipe data sesudah:
Year    int64
dtype: object

Kolom Year berhasil diubah dari float64 → int64.


In [ ]:
# 6b. Pastikan kolom sales bertipe float dan tidak negatif
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']

for col in sales_cols:
    df_clean[col] = df_clean[col].astype(float)
    # Ganti nilai negatif (jika ada) dengan 0
    negatif = (df_clean[col] < 0).sum()
    if negatif > 0:
        df_clean[col] = df_clean[col].clip(lower=0)
        print(f'   {col}: {negatif} nilai negatif diganti 0')

print("Semua kolom sales sudah bertipe float dan tidak negatif.")

Semua kolom sales sudah bertipe float dan tidak negatif.


## 7. Feature Engineering

Membuat kolom baru yang berguna untuk visualisasi dashboard.

In [ ]:
# 7a. Kolom Era / Dekade
df_clean['Era'] = pd.cut(
    df_clean['Year'],
    bins=[1979, 1989, 1999, 2009, 2016],
    labels=['80s (1980–1989)', '90s (1990–1999)', '2000s (2000–2009)', '2010s (2010–2016)']
)

print("Kolom Era berhasil dibuat.")
print(df_clean['Era'].value_counts())

Kolom Era berhasil dibuat.
Era
2000s (2000–2009)    9183
2010s (2010–2016)    5130
90s (1990–1999)      1769
80s (1980–1989)       205
Name: count, dtype: int64


In [17]:
# 7b. Kolom Dominant Region
# Region mana yang paling banyak menyumbang penjualan tiap game
df_clean['Dominant_Region'] = df_clean[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].idxmax(axis=1)
df_clean['Dominant_Region'] = df_clean['Dominant_Region'].str.replace('_Sales', '')

print("\nKolom Dominant_Region berhasil dibuat.")
print(df_clean['Dominant_Region'].value_counts())


Kolom Dominant_Region berhasil dibuat.
Dominant_Region
NA       9898
JP       3981
EU       2332
Other      76
Name: count, dtype: int64


In [18]:
# 7c. Kolom Sales Category (tingkatan penjualan)
df_clean['Sales_Category'] = pd.cut(
    df_clean['Global_Sales'],
    bins=[0, 0.5, 1, 5, 10, 100],
    labels=['Very Low (<0.5M)', 'Low (0.5–1M)', 'Medium (1–5M)', 'High (5–10M)', 'Blockbuster (>10M)']
)

print("Kolom Sales_Category berhasil dibuat.")
print(df_clean['Sales_Category'].value_counts())

Kolom Sales_Category berhasil dibuat.
Sales_Category
Very Low (<0.5M)      12397
Low (0.5–1M)           1859
Medium (1–5M)          1827
High (5–10M)            142
Blockbuster (>10M)       62
Name: count, dtype: int64


## 8. Filtering Data Tidak Relevan

Menghapus data yang tidak berkontribusi pada analisis.

In [19]:
sebelum = len(df_clean)

# Hapus game dengan Global_Sales = 0 (tidak ada penjualan sama sekali)
df_clean = df_clean[df_clean['Global_Sales'] > 0]

# Hapus kolom Rank karena tidak relevan untuk analisis visual
df_clean = df_clean.drop(columns=['Rank'])

# Reset index
df_clean = df_clean.reset_index(drop=True)

print("Filtering selesai.")
print(f"Baris sebelum : {sebelum:,}")
print(f"Baris sesudah : {len(df_clean):,}")
print(f"Kolom Rank    : dihapus (tidak relevan untuk visualisasi)")
print(f"\nKolom akhir: {list(df_clean.columns)}")

Filtering selesai.
Baris sebelum : 16,287
Baris sesudah : 16,287
Kolom Rank    : dihapus (tidak relevan untuk visualisasi)

Kolom akhir: ['Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Era', 'Dominant_Region', 'Sales_Category']


## 9. Hasil Akhir & Export

Menampilkan perbandingan kondisi data sebelum dan sesudah preprocessing, lalu menyimpan hasil.

In [20]:
print("=" * 55)
print("RINGKASAN HASIL PREPROCESSING")
print("=" * 55)
print(f"Baris awal          : 16,598")
print(f"Baris akhir         : {len(df_clean):,}")
print(f"Baris terhapus      : {16598 - len(df_clean):,}")
print(f"Kolom awal          : 11")
print(f"Kolom akhir         : {df_clean.shape[1]}")
print(f"Missing value       : 0")
print(f"Duplikat            : 0")
print(f"Kolom baru (FE)     : Era, Dominant_Region, Sales_Category")
print("=" * 55)
print()
print("Preview 5 baris pertama data bersih:")
df_clean.head()

RINGKASAN HASIL PREPROCESSING
Baris awal          : 16,598
Baris akhir         : 16,287
Baris terhapus      : 311
Kolom awal          : 11
Kolom akhir         : 13
Missing value       : 0
Duplikat            : 0
Kolom baru (FE)     : Era, Dominant_Region, Sales_Category

Preview 5 baris pertama data bersih:


,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Era,Dominant_Region,Sales_Category
0,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74,2000s (2000–2009),NA,Blockbuster (>10M)
1,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,80s (1980–1989),NA,Blockbuster (>10M)
2,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82,2000s (2000–2009),NA,Blockbuster (>10M)
3,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00,2000s (2000–2009),NA,Blockbuster (>10M)
4,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,90s (1990–1999),NA,Blockbuster (>10M)


In [21]:
# Tampilkan info tipe data final
print('Tipe data kolom akhir:')
print(df_clean.dtypes)
print()
print('Cek missing value akhir:')
print(df_clean.isnull().sum())

Tipe data kolom akhir:
Name                    str
Platform                str
Year                  int64
Genre                   str
Publisher               str
NA_Sales            float64
EU_Sales            float64
JP_Sales            float64
Other_Sales         float64
Global_Sales        float64
Era                category
Dominant_Region         str
Sales_Category     category
dtype: object

Cek missing value akhir:
Name               0
Platform           0
Year               0
Genre              0
Publisher          0
NA_Sales           0
EU_Sales           0
JP_Sales           0
Other_Sales        0
Global_Sales       0
Era                0
Dominant_Region    0
Sales_Category     0
dtype: int64


In [22]:
# Simpan ke file processed
output_path = '../data/processed/video_game_sales_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f"Data bersih berhasil disimpan ke: {output_path}")
print(f"Shape final: {df_clean.shape}")

Data bersih berhasil disimpan ke: ../data/processed/video_game_sales_clean.csv
Shape final: (16287, 13)


---
## Kesimpulan Preprocessing

| Tahap | Aksi | Hasil |
|-------|------|-------|
| Missing Value | Hapus baris dengan `Year` & `Publisher` kosong | −329 baris |
| Duplikat | Cek dan hapus duplikat | Sesuai hasil pengecekan |
| Konsistensi | Strip whitespace, cek rentang Year (1980–2016) | Data lebih bersih |
| Format Data | `Year` float → int, sales pastikan tidak negatif | Tipe data sesuai |
| Feature Engineering | Tambah kolom `Era`, `Dominant_Region`, `Sales_Category` | +3 kolom baru |
| Filtering | Hapus sales = 0, hapus kolom `Rank` | Data lebih relevan |

> Dataset bersih siap digunakan untuk pembuatan visualisasi interaktif di dashboard.